In [ ]:
%load_ext autoreload
%autoreload 2

**Author:** Salvador Navas  
**Date:** 2025-06-27

### SoilGrids — Global Soil Information (ISRIC)

**SoilGrids** is a global system for digital soil mapping developed by ISRIC – World Soil Information. It provides predictions of soil properties at multiple depths worldwide, derived from machine learning models trained on field observations and remote sensing covariates.

#### 💡 **Key characteristics:**

- **Provider:** ISRIC – World Soil Information ([https://www.isric.org](https://www.isric.org)).
- **Dataset:** SoilGrids 250m (2017 release) — sand, silt, and clay percentage at 7 standard depths.
- **Variables available:** Sand (`SNDPPT`), Silt (`SLTPPT`), Clay (`CLYPPT`) — percent by weight.
- **Depth layers:** sl1–sl7 corresponding to 0, 5, 15, 30, 60, 100, 200 cm.
- **Spatial resolution:** 250 m (~0.002°), global coverage.
- **Format:** GeoTIFF (uint8, values 0–100 = percentage; 255 = no data).
- **License:** CC BY 4.0 — free for any use with attribution.

#### 🖥️ **Workflow overview:**

| Step | Function | Output |
|------|----------|--------|
| 1 — Download GeoTIFFs | `download_soilgrids` | 21 `.tif` files (3 vars × 7 depths) |
| 2 — USDA classification | `convert_depth_to_soilclass` | 7 USDA class GeoTIFFs |
| 3 — Modal depth aggregation | `extract_mode_soilclass` | 1 dominant soil-class GeoTIFF |

#### 🔗 **References:**

- Hengl T. et al. (2017). *SoilGrids250m: Global gridded soil information based on machine learning*. PLoS ONE 12(2): e0169748.
- ISRIC data portal: [https://files.isric.org/soilgrids/former/2017-03-10/data/](https://files.isric.org/soilgrids/former/2017-03-10/data/)

In [ ]:
from pyhydra.data_sources.soils import (
    download_soilgrids,
    open_soilgrids_geotif,
    find_usda_soilclass,
    convert_depth_to_soilclass,
    extract_mode_soilclass,
)
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import os
from pathlib import Path

In [ ]:
# === SETUP WORKING DIRECTORY ===

# Automatically detect the path where the notebook is located
notebook_dir = Path().resolve()
os.chdir(notebook_dir)

# === GENERAL PARAMETERS ===

# Output directory for raw SoilGrids GeoTIFFs
RAW_DIR   = 'soilgrids_raw/'

# Output directory for USDA soil-class GeoTIFFs (one per depth)
CLASS_DIR = 'soilgrids_usda/'

# Output file for the modal soil class across all depths
MODE_FILE = 'soilgrids_usda/usda_soilclass_mode.tif'

# Download also XML and CSV metadata files (optional)
INCLUDE_METADATA = False

# USDA soil texture class names (class 1–12)
SOIL_TYPES = [
    'Clay', 'Clay Loam', 'Loam', 'Loamy Sand', 'Sand',
    'Sandy Clay', 'Sandy Clay Loam', 'Sandy Loam',
    'Silt', 'Silty Clay', 'Silty Clay Loam', 'Silt Loam',
]

---
## 1. ⬇️ Download SoilGrids GeoTIFFs

Downloads **21 GeoTIFF files** (sand, silt, clay × 7 depths) from the ISRIC public server. No registration required. Files already present are skipped automatically.

In [ ]:
download_soilgrids(
    output_dir=RAW_DIR,
    max_retries=10,
    include_metadata=INCLUDE_METADATA,
)

tif_files = sorted(Path(RAW_DIR).glob('*.tif'))
print(f'GeoTIFFs available: {len(tif_files)}')
for f in tif_files:
    print(' ', f.name)

---
## 2. 🗺️ Inspect a single GeoTIFF

Open one layer and visualise the raw texture percentage.

In [ ]:
# Open sand content at surface layer (sl1 = 0–5 cm)
LAYER_FILE = Path(RAW_DIR) / 'SNDPPT_M_sl1_250m_ll.tif'

sand_data, coords = open_soilgrids_geotif(LAYER_FILE)
print(f'Array shape : {sand_data.shape}')
print(f'Bounding box: N={coords[0,1]:.2f}  W={coords[0,0]:.2f}  S={coords[3,1]:.2f}  E={coords[3,0]:.2f}')

# Replace no-data (255) with NaN for display
sand_plot = sand_data.astype(float)
sand_plot[sand_plot == 255] = np.nan

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(sand_plot, cmap='YlOrBr', vmin=0, vmax=100,
               extent=[coords[0,0], coords[2,0], coords[3,1], coords[0,1]])
plt.colorbar(im, ax=ax, label='Sand content (%)', fraction=0.03, pad=0.02)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('SoilGrids — Sand content 0–5 cm (sl1)', fontsize=13)
plt.tight_layout()
plt.show()

---
## 3. 🧪 USDA soil texture classification

Classify each pixel into one of 12 **USDA soil texture classes** (Clay, Loam, Sand, etc.) using the sand/silt/clay percentages from a single depth layer.

In [ ]:
# Classify a single depth layer
DEPTH = 'sl1'   # change to sl2..sl7 for other depths

sand, coords = open_soilgrids_geotif(Path(RAW_DIR) / f'SNDPPT_M_{DEPTH}_250m_ll.tif')
silt, _      = open_soilgrids_geotif(Path(RAW_DIR) / f'SLTPPT_M_{DEPTH}_250m_ll.tif')
clay, _      = open_soilgrids_geotif(Path(RAW_DIR) / f'CLYPPT_M_{DEPTH}_250m_ll.tif')

soilclass, soiltype = find_usda_soilclass(sand, silt, clay)

# Class frequency
unique, counts = np.unique(soilclass[soilclass > 0], return_counts=True)
print(f'USDA classes present at depth {DEPTH}:')
for cls, cnt in sorted(zip(unique, counts), key=lambda x: -x[1]):
    print(f'  {cls:2d} — {soiltype[cls-1]:<20s}: {cnt:,} pixels ({100*cnt/soilclass.size:.1f} %)')

In [ ]:
# Visualise the USDA classification at this depth
cmap = plt.cm.get_cmap('tab20', 13)   # 12 classes + 0 (no data)
sc_plot = soilclass.astype(float)
sc_plot[sc_plot == 0] = np.nan

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(sc_plot, cmap=cmap, vmin=0.5, vmax=12.5,
               extent=[coords[0,0], coords[2,0], coords[3,1], coords[0,1]])
cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02, ticks=range(1, 13))
cbar.ax.set_yticklabels(SOIL_TYPES, fontsize=8)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'USDA Soil Texture Classification — depth {DEPTH} (0–5 cm)', fontsize=13)
plt.tight_layout()
plt.show()

---
## 4. 📂 Convert all depths to USDA classes

Process all 7 depth layers and save one USDA soil-class GeoTIFF per depth. Requires GDAL (`pip install gdal` or `conda install gdal`).

In [ ]:
convert_depth_to_soilclass(
    input_dir=RAW_DIR,
    output_dir=CLASS_DIR,
)

class_files = sorted(Path(CLASS_DIR).glob('usda_soilclass_sl*.tif'))
print(f'\nUSDA class GeoTIFFs written: {len(class_files)}')
for f in class_files:
    print(' ', f.name)

---
## 5. 🏆 Modal soil class across all depths

Compute the **dominant (modal) USDA soil class** across the 7 depth layers — the class that appears most frequently through the soil profile at each pixel.

In [ ]:
extract_mode_soilclass(
    input_dir=CLASS_DIR,
    output_path=MODE_FILE,
)

print(f'\nMode GeoTIFF saved to: {MODE_FILE}')

In [ ]:
# Visualise the dominant soil class map
mode_data, coords_m = open_soilgrids_geotif(MODE_FILE)
mode_plot = mode_data.astype(float)
mode_plot[mode_plot == 0] = np.nan

# Class frequency summary
unique_m, counts_m = np.unique(mode_data[mode_data > 0], return_counts=True)
print('Dominant USDA soil classes (global):')
for cls, cnt in sorted(zip(unique_m, counts_m), key=lambda x: -x[1]):
    name = SOIL_TYPES[int(cls) - 1] if int(cls) <= len(SOIL_TYPES) else str(cls)
    print(f'  {int(cls):2d} — {name:<20s}: {cnt:,} pixels ({100*cnt/mode_data.size:.1f} %)')

cmap = plt.cm.get_cmap('tab20', 13)
fig, ax = plt.subplots(figsize=(16, 7))
im = ax.imshow(mode_plot, cmap=cmap, vmin=0.5, vmax=12.5,
               extent=[coords_m[0,0], coords_m[2,0], coords_m[3,1], coords_m[0,1]])
cbar = plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02, ticks=range(1, 13))
cbar.ax.set_yticklabels(SOIL_TYPES, fontsize=8)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Dominant USDA Soil Texture Class — modal across 7 depth layers', fontsize=13)
plt.tight_layout()
plt.show()